# ATLAS Wind Downscaling

This notebook performs wind downscaling starting from the preprocessed ERA5Land wind component data.

It combines the previous islands and continental workflows into one single notebook. The same functions are used as much as possible for both cases. The only operational difference is the area strategy:

* `AREA_MODE = "islands"` runs the selected bbox in one piece.
* `AREA_MODE = "continental"` trains the model on the full continental bbox, then splits the GLO90 prediction step into smaller latitude bands according to the bbox.

Run the notebook from top to bottom. Most users should only edit the configuration cell.

## Methodology Description:

This notebook applies a Machine Learning based statistical downscaling approach to generate high-resolution climate fields from coarse-resolution climate datasets. The method relies on a Multi-Layer Perceptron (MLP) neural network trained to learn the relationship between large-scale climate variables and local terrain characteristics.

The downscaling procedure uses a set of predictors describing the influence of topography on local climate conditions. These predictors are derived from the Copernicus GLO90 Digital Elevation Model and typically include elevation, slope, aspect, and other terrain-related features. The target variable depends on the application and may include temperature, precipitation, solar radiation, or wind components.

During the training phase, the MLP is calibrated using the coarse-resolution climate variable together with the corresponding topographic predictors. The trained model is then applied to the high-resolution GLO90 grid, allowing the generation of climate information at a much finer spatial resolution.

The use of Earth Observation data is a key element of the methodology. High-resolution terrain information derived from satellite observations provides detailed spatial descriptors that are not represented in global climate datasets, enabling the neural network to reproduce local-scale spatial variability driven by topography.

This approach preserves the large-scale climate signal provided by reanalysis or climate model data while enhancing its spatial detail, producing high-resolution climate layers suitable for local impact assessments and climate adaptation studies.

## 1. Import libraries

The notebook requires the same Python environment used for the previous Solar and Wind workflows. In particular, it needs `xarray`, `rioxarray`, `pandas`, `numpy`, `scikit-learn`, `netCDF4`, and optionally `geopandas` and `matplotlib` for plotting.


In [1]:
from pathlib import Path
import gc
import os

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
import numpy as np
import xarray as xr
import pandas as pd
import rioxarray

try:
    import geopandas as gpd
    import matplotlib.pyplot as plt
except ImportError:
    gpd = None
    plt = None


## 2. User configuration

Edit only this cell before running the notebook.

The bbox format is always:

`[lat_max, lon_min, lat_min, lon_max]`

For Chile, latitudes are negative. `lat_max` is therefore the northern boundary, while `lat_min` is the southern boundary.


In [2]:
# General settings
country = "chile"
month = 1
comp = "u"  # Choose "u" or "v"

# Choose the processing strategy.
# Use "islands" for island boxes or small domains.
# Use "continental" for large continental domains that should be processed in chunks.
AREA_MODE = "continental"

# Bounding boxes
BBOXES = {
    "continental": [-17.3, -76.2, -56.7, -66.2],
    "islands": [-26.0, -109.8, -34.8, -77.8],
}

# Select the bbox to process.
# For AREA_MODE = "continental", keep AREA_NAME = "continental" unless you know what you are doing.
# For AREA_MODE = "islands", you can use "islands" or one of the single island boxes above.
# AREA_NAME follows AREA_MODE by default.
# Set it manually only if you want to process a specific named bbox.
AREA_NAME = AREA_MODE
AREA_BBOX = BBOXES[AREA_NAME]

# Continental split settings.
# The model is trained once on the full bbox.
# The prediction on the GLO90 grid is split into latitude bands to reduce memory usage.
CONTINENTAL_N_SPLITS = 2
CONTINENTAL_BUFFER_DEG = 1.0

# Generic project paths.
# Keep these paths relative when possible, so the notebook can run on different machines.
PROJECT_DIR = Path("../data")
PROCESSED_WIND_DIR = PROJECT_DIR / "processed" / f"10m_wind_{comp}_component" / country

DEM_DIR =  Path("../DEMdata") / country
IRI_DIR = PROJECT_DIR / "processed" / f'{comp}_clim'
INTERMEDIATE_DIR = PROJECT_DIR / "intermediate_steps" / country
OUTPUT_DIR = PROJECT_DIR / "downscaled_data" / country / "sub_areas"
SUBAREA_OUTPUT_DIR = OUTPUT_DIR / "sub_areas"

# Input file names.
# Change these only if your preprocessing notebook created different names.
PROCESSED_COMPONENT_FILE = PROCESSED_WIND_DIR / f"{comp}10_1991-01_2020-12_processed.nc"
ERA5_OROGRAPHY_FILE = DEM_DIR / f"era5land_orography_{country}.nc"
ERA5_ASPECT_FILE = DEM_DIR / f"era5land_aspect_{country}.nc"
GLO90_OROGRAPHY_FILE = DEM_DIR / f"glo90_orography_{country}.nc"
GLO90_ASPECT_FILE = DEM_DIR / f"glo90_aspect_{country}.nc"
IRI_COMPONENT_FILE = IRI_DIR / f"{comp}_iri_processed_global.nc"

# Optional shapefile for visual checks.
# Leave as None if you do not need plotting.
SHAPEFILE_PATH = None

# Model settings
MLP_RANDOM_STATE = 1
MLP_MAX_ITER = 1000


## 3. Create output folders and check inputs

This cell creates the output folders and checks that the required input files exist. If a file is missing, update the paths in the configuration cell.


In [3]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBAREA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    PROCESSED_COMPONENT_FILE,
    ERA5_OROGRAPHY_FILE,
    ERA5_ASPECT_FILE,
    GLO90_OROGRAPHY_FILE,
    GLO90_ASPECT_FILE,
    IRI_COMPONENT_FILE,
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    print("Missing input files:")
    for path in missing_files:
        print(f"  {path}")
    raise FileNotFoundError("Please update the paths in the configuration cell before continuing.")

print("All required input files were found.")
print(f"Area mode: {AREA_MODE}")
print(f"Area name: {AREA_NAME}")
print(f"Area bbox: {AREA_BBOX}")


All required input files were found.
Area mode: continental
Area name: continental
Area bbox: [-17.3, -76.2, -56.7, -66.2]


## 4. Helper functions

These functions are shared by the islands and continental workflows.


In [4]:
def drop_spatial_ref(obj):
    """Drop common CRS helper variables when they are present."""
    return obj.drop_vars(["spatial_ref", "crs"], errors="ignore")


def save_xarray_netcdf_fast(ds, output_path):
    """Save an xarray Dataset as compressed NetCDF."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    encoding = {
        var: {"zlib": True, "complevel": 1}
        for var in ds.data_vars
    }

    ds.to_netcdf(
        output_path,
        engine="netcdf4",
        encoding=encoding,
    )


def validate_bbox(bbox):
    """Validate bbox format: [lat_max, lon_min, lat_min, lon_max]."""
    if len(bbox) != 4:
        raise ValueError("BBOX must have four values: [lat_max, lon_min, lat_min, lon_max].")

    lat_max, lon_min, lat_min, lon_max = bbox
    if lat_max <= lat_min:
        raise ValueError("lat_max must be greater than lat_min. Remember that Chile latitudes are negative.")
    if lon_min >= lon_max:
        raise ValueError("lon_min must be smaller than lon_max.")


def split_bbox_by_latitude(bbox, n_splits=2, buffer_deg=0.0):
    """Split a bbox into latitude bands, keeping the same longitude range.

    The returned boxes overlap by buffer_deg. This avoids edge artefacts when merging the chunks.
    """
    validate_bbox(bbox)
    lat_max, lon_min, lat_min, lon_max = bbox

    if n_splits < 1:
        raise ValueError("n_splits must be at least 1.")

    if n_splits == 1:
        return {"full": bbox}

    edges = np.linspace(lat_max, lat_min, n_splits + 1)
    subareas = {}

    for idx in range(n_splits):
        north_edge = edges[idx]
        south_edge = edges[idx + 1]

        sub_lat_max = min(lat_max, north_edge + buffer_deg if idx > 0 else north_edge)
        sub_lat_min = max(lat_min, south_edge - buffer_deg if idx < n_splits - 1 else south_edge)

        if n_splits == 2:
            label = "north" if idx == 0 else "south"
        else:
            label = f"band_{idx + 1:02d}"

        subareas[label] = [float(sub_lat_max), lon_min, float(sub_lat_min), lon_max]

    return subareas


def select_bbox(ds, bbox):
    """Select a bbox from an xarray object using latitude and longitude.

    The notebook uses bbox format [lat_max, lon_min, lat_min, lon_max].
    This helper works with both descending and ascending latitude coordinates,
    and with both ascending and descending longitude coordinates.
    """
    validate_bbox(bbox)
    lat_max, lon_min, lat_min, lon_max = bbox

    lat = ds["latitude"]
    lon = ds["longitude"]

    lat_slice = slice(lat_max, lat_min) if lat[0] > lat[-1] else slice(lat_min, lat_max)
    lon_slice = slice(lon_min, lon_max) if lon[0] < lon[-1] else slice(lon_max, lon_min)

    selected = ds.sel(
        longitude=lon_slice,
        latitude=lat_slice,
    )

    if selected.sizes.get("latitude", 0) == 0 or selected.sizes.get("longitude", 0) == 0:
        print("Warning: selected bbox has an empty grid.")
        print(f"  bbox: {bbox}")
        print(f"  latitude range in dataset: {float(lat.min())} to {float(lat.max())}")
        print(f"  longitude range in dataset: {float(lon.min())} to {float(lon.max())}")

    return selected


def add_aspect_sin_cos(df):
    """Convert aspect degrees into sine and cosine features."""
    df = df.copy()

    if "aspect" in df.columns:
        df["aspect_sin"] = np.sin(np.deg2rad(df["aspect"]))
        df["aspect_cos"] = np.cos(np.deg2rad(df["aspect"]))
        df = df.drop(columns=["aspect"])

    return df


## 5. Machine learning functions

The same training and prediction functions are used for islands and continental areas.


In [5]:
def era5scaler(xdf, features, target):
    """Scale ERA5Land features and target using StandardScaler."""
    featurescaler = StandardScaler()
    X_scaled = featurescaler.fit_transform(xdf[features])

    targetscaler = StandardScaler()
    y_scaled = targetscaler.fit_transform(xdf[[target]]).ravel()

    return featurescaler, targetscaler, X_scaled, y_scaled


def make_dataset_era5land(target_var, orography, aspect, iri_ds):
    """Create the training dataframe by merging ERA5Land target, DEM features and IRI data."""
    target_var_monthly = target_var.groupby(target_var.time.dt.month).mean()

    xdf_merged = xr.merge([
        orography,
        aspect,
        iri_ds,
        target_var_monthly,
    ])

    merged = xdf_merged.to_dataframe().dropna().reset_index()
    return merged


def training_ds(era5l_merged, month, target):
    """Prepare the monthly training dataframe and train the MLP model."""
    era5l_merged = (
        era5l_merged
        .loc[era5l_merged.month == month, :]
        .drop("month", axis=1)
    )

    era5l_merged = add_aspect_sin_cos(era5l_merged)

    features = era5l_merged.columns.drop([
        target,
        "latitude",
        "longitude",
    ])

    featurescaler_era5, targetscaler_era5, era5l_X, era5l_y = era5scaler(
        era5l_merged,
        features,
        target,
    )

    regr = MLPRegressor(
        random_state=MLP_RANDOM_STATE,
        max_iter=MLP_MAX_ITER,
    ).fit(era5l_X, era5l_y)

    return features, featurescaler_era5, targetscaler_era5, era5l_X, era5l_y, regr


def apply_downscaling(dataset_glo90, features, template_da, comp, featurescaler_era5, targetscaler_era5, regr):
    """Apply the trained model to the GLO90 dataframe and map predictions back to the grid."""
    df = dataset_glo90.copy()
    df = add_aspect_sin_cos(df)

    glo90_X = featurescaler_era5.transform(df[features])
    downscaled_std = regr.predict(glo90_X)
    downscaled = targetscaler_era5.inverse_transform(
        downscaled_std.reshape(-1, 1)
    ).ravel()

    out = xr.full_like(template_da, np.nan, dtype=float)
    out_values = out.values.copy()

    lat_idx = template_da.get_index("latitude").get_indexer(df["latitude"].to_numpy())
    lon_idx = template_da.get_index("longitude").get_indexer(df["longitude"].to_numpy())

    missing = (lat_idx < 0) | (lon_idx < 0)
    if missing.any():
        raise ValueError(
            f"Some dataframe coordinates were not found in the template grid: {missing.sum()} missing points. "
            "Check latitude and longitude precision."
        )

    out_values[lat_idx, lon_idx] = downscaled
    out.values[:] = out_values

    return xr.Dataset({
        f"{comp}10_downscaled": out,
    })


## 6. Data loading functions

These functions load the preprocessed wind component, ERA5Land DEM variables, GLO90 DEM variables and IRI monthly fields.


In [6]:
def load_input_datasets():
    """Load all input datasets and align them to the grids used by the workflow."""
    print("Opening and preparing ERA5Land data...")

    component = xr.open_mfdataset(PROCESSED_COMPONENT_FILE).sel(latitude=slice(None, None, -1))
    component = drop_spatial_ref(component.rio.write_crs("EPSG:4326"))

    orography = xr.open_dataset(ERA5_OROGRAPHY_FILE)
    orography = drop_spatial_ref(orography.rio.write_crs("EPSG:4326"))

    aspect = xr.open_dataset(ERA5_ASPECT_FILE)
    aspect = drop_spatial_ref(aspect.rio.write_crs("EPSG:4326"))

    comp_iri = xr.open_dataset(IRI_COMPONENT_FILE)
    comp_iri = drop_spatial_ref(comp_iri)

    print("Interpolating ERA5Land static fields to the wind component grid...")
    orography_interp = orography.interp(
        latitude=component.latitude.values,
        longitude=component.longitude.values,
        method="nearest",
    )
    aspect_interp = aspect.interp(
        latitude=component.latitude.values,
        longitude=component.longitude.values,
        method="nearest",
    )
    iri_ds_interp = comp_iri.interp(
        latitude=component.latitude,
        longitude=component.longitude,
        method="slinear",
    )

    print("Opening and preparing GLO90 data...")
    orography_glo90 = xr.open_dataset(GLO90_OROGRAPHY_FILE)
    orography_glo90 = drop_spatial_ref(orography_glo90.rio.write_crs("EPSG:4326"))

    aspect_glo90 = xr.open_dataset(GLO90_ASPECT_FILE)
    aspect_glo90 = drop_spatial_ref(aspect_glo90.rio.write_crs("EPSG:4326"))

    aspect_glo90_interp = aspect_glo90.interp(
        latitude=orography_glo90.latitude.values,
        longitude=orography_glo90.longitude.values,
        method="nearest",
    )

    print("Preparing monthly IRI field on the GLO90 grid...")
    comp_iri_month = comp_iri.sel(month=month).drop_vars("month", errors="ignore")
    iri_month_file = INTERMEDIATE_DIR / f"{comp}_iri_monthly_mean_{country}_month{month}.nc"
    save_xarray_netcdf_fast(comp_iri_month, iri_month_file)

    comp_iri_month = xr.open_dataset(iri_month_file)
    comp_iri_glo90_interp = comp_iri_month.interp(
        latitude=orography_glo90.latitude.values,
        longitude=orography_glo90.longitude.values,
        method="slinear",
    )

    return {
        "component": component,
        "orography_interp": orography_interp,
        "aspect_interp": aspect_interp,
        "iri_ds_interp": iri_ds_interp,
        "orography_glo90": orography_glo90,
        "aspect_glo90_interp": aspect_glo90_interp,
        "comp_iri_glo90_interp": comp_iri_glo90_interp,
    }


## 7. Downscaling workflow functions

The continental workflow trains once on the full bbox and predicts by latitude chunks. The islands workflow trains and predicts on the selected bbox without chunking.


In [7]:
def train_model_for_bbox(datasets, bbox):
    """Train the downscaling model on the ERA5Land grid for the selected bbox."""
    print("Preparing ERA5Land training dataset...")

    orography_area = select_bbox(datasets["orography_interp"], bbox)
    aspect_area = select_bbox(datasets["aspect_interp"], bbox)
    iri_area = select_bbox(datasets["iri_ds_interp"], bbox)
    component_area = select_bbox(datasets["component"], bbox)

    era5l_merged = make_dataset_era5land(
        component_area,
        orography_area,
        aspect_area,
        iri_area,
    )

    target = f"{comp}10"

    print("Training model...")
    features, featurescaler_era5, targetscaler_era5, era5l_X, era5l_y, regr = training_ds(
        era5l_merged,
        month,
        target,
    )

    print("Model features:", list(features))

    return {
        "features": features,
        "featurescaler_era5": featurescaler_era5,
        "targetscaler_era5": targetscaler_era5,
        "regr": regr,
    }


def make_glo90_dataframe_for_bbox(datasets, bbox):
    """Create the GLO90 prediction dataframe for the selected bbox."""
    orography_glo90_area = select_bbox(datasets["orography_glo90"], bbox)
    aspect_glo90_area = select_bbox(datasets["aspect_glo90_interp"], bbox)
    iri_glo90_area = select_bbox(datasets["comp_iri_glo90_interp"], bbox)

    oro = drop_spatial_ref(orography_glo90_area["z"])
    asp = drop_spatial_ref(aspect_glo90_area["aspect"])
    iri = drop_spatial_ref(iri_glo90_area[comp])

    oro, asp, iri = xr.align(oro, asp, iri, join="exact")

    valid_mask = oro.notnull() & asp.notnull()
    oro = oro.where(valid_mask)
    asp = asp.where(valid_mask)
    iri = iri.where(valid_mask)

    glo90_ds = xr.Dataset({
        "z": oro,
        "aspect": asp,
        comp: iri,
    })

    dataset_glo90 = (
        glo90_ds
        .to_dataframe()
        .dropna()
        .reset_index()
    )

    print(f"Valid GLO90 pixels: {len(dataset_glo90)}")
    return glo90_ds, dataset_glo90


def downscale_prediction_bbox(datasets, model, bbox):
    """Apply the trained model to one GLO90 bbox."""
    glo90_ds, dataset_glo90 = make_glo90_dataframe_for_bbox(datasets, bbox)

    if dataset_glo90.empty:
        print("No valid GLO90 pixels were found for this bbox. The chunk will be skipped.")
        return None

    output_ds = apply_downscaling(
        dataset_glo90,
        model["features"],
        glo90_ds["z"],
        comp,
        model["featurescaler_era5"],
        model["targetscaler_era5"],
        model["regr"],
    )

    return output_ds


def run_islands_workflow(datasets, area_name, bbox):
    """Run the full workflow for islands or small domains without splitting."""
    print(f"Running islands workflow for {area_name}...")
    model = train_model_for_bbox(datasets, bbox)
    output_ds = downscale_prediction_bbox(datasets, model, bbox)

    output_file = SUBAREA_OUTPUT_DIR / f"{comp}10m_component_downscaled_{country}_m{month}_{area_name}.nc"
    print(f"Saving: {output_file}")
    save_xarray_netcdf_fast(output_ds, output_file)

    return output_ds, output_file


def run_continental_workflow(datasets, area_name, bbox):
    """Run the continental workflow.

    The model is trained once on the full continental bbox.
    The prediction is split into latitude bands to reduce memory usage.
    The chunks are saved separately and then merged into one final output file.
    """
    print(f"Running continental workflow for {area_name}...")
    print("Training uses the full continental bbox.")

    model = train_model_for_bbox(datasets, bbox)

    subareas = split_bbox_by_latitude(
        bbox,
        n_splits=CONTINENTAL_N_SPLITS,
        buffer_deg=CONTINENTAL_BUFFER_DEG,
    )

    output_files = []

    for subarea_name, subarea_bbox in subareas.items():
        print(f"Processing continental subarea: {subarea_name}")
        print(f"Subarea bbox: {subarea_bbox}")

        output_ds = downscale_prediction_bbox(datasets, model, subarea_bbox)

        if output_ds is None:
            continue

        output_file = SUBAREA_OUTPUT_DIR / f"{comp}10m_component_downscaled_{country}_m{month}_{area_name}_{subarea_name}.nc"
        print(f"Saving: {output_file}")
        save_xarray_netcdf_fast(output_ds, output_file)
        output_files.append(output_file)

        del output_ds
        gc.collect()

    if not output_files:
        raise ValueError(
            "No continental chunk produced valid output. Check AREA_BBOX, longitude convention and input files."
        )
    
    print("Merging continental subareas...")
    
    merged = None
    
    for output_file in output_files:
        print(f"Opening chunk: {output_file}")
    
        ds = xr.open_dataset(
            output_file,
            chunks={
                "latitude": 1000,
                "longitude": 1000,
            },
        )
    
        ds = drop_spatial_ref(ds.rio.write_crs("EPSG:4326"))
    
        if "latitude" in ds.coords:
            ds = ds.sortby("latitude")
        if "longitude" in ds.coords:
            ds = ds.sortby("longitude")
    
        if merged is None:
            merged = ds
        else:
            merged = merged.combine_first(ds)
    
    print("Sorting final grid...")
    
    if "latitude" in merged.coords:
        merged = merged.sortby("latitude")
    if "longitude" in merged.coords:
        merged = merged.sortby("longitude")
    
    final_output_file = OUTPUT_DIR / f"{comp}10m_component_downscaled_{country}_m{month}_{area_name}.nc"
    
    print(f"Saving merged continental output: {final_output_file}")
    save_xarray_netcdf_fast(merged, final_output_file)

    return merged, final_output_file, output_files


## 8. Run the downscaling

This cell chooses the correct workflow based on `AREA_MODE`.

Use `AREA_MODE = "islands"` for a single small bbox. Use `AREA_MODE = "continental"` when the prediction should be split into latitude bands.


In [8]:
validate_bbox(AREA_BBOX)

datasets = load_input_datasets()

if AREA_MODE == "islands":
    output_ds, final_output_file = run_islands_workflow(
        datasets,
        AREA_NAME,
        AREA_BBOX,
    )
    chunk_output_files = []

elif AREA_MODE == "continental":
    output_ds, final_output_file, chunk_output_files = run_continental_workflow(
        datasets,
        AREA_NAME,
        AREA_BBOX,
    )

else:
    raise ValueError("AREA_MODE must be either 'islands' or 'continental'.")

print("Done.")
print(f"Final output: {final_output_file}")

if chunk_output_files:
    print("Chunk outputs:")
    for path in chunk_output_files:
        print(f"  {path}")


Opening and preparing ERA5Land data...


/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.31.0 or higher is recommended. You are running version 2.16.0
  warnings.warn(
ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed


Interpolating ERA5Land static fields to the wind component grid...
Opening and preparing GLO90 data...
Preparing monthly IRI field on the GLO90 grid...
Running continental workflow for continental...
Training uses the full continental bbox.
Preparing ERA5Land training dataset...
Training model...
Model features: ['z', 'u', 'aspect_sin', 'aspect_cos']
Processing continental subarea: north
Subarea bbox: [-17.3, -76.2, -38.0, -66.2]
Valid GLO90 pixels: 55387205
Saving: ../data/downscaled_data/chile/sub_areas/sub_areas/u10m_component_downscaled_chile_m1_continental_north.nc
Processing continental subarea: south
Subarea bbox: [-36.0, -76.2, -56.7, -66.2]
Valid GLO90 pixels: 65209480
Saving: ../data/downscaled_data/chile/sub_areas/sub_areas/u10m_component_downscaled_chile_m1_continental_south.nc
Merging continental subareas...
Opening chunk: ../data/downscaled_data/chile/sub_areas/sub_areas/u10m_component_downscaled_chile_m1_continental_north.nc
Opening chunk: ../data/downscaled_data/chile/s

/tmp/ipykernel_1850018/799403570.py:153: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 1000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_1850018/799403570.py:153: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 1000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_1850018/799403570.py:153: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 1000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(
/tmp/ipykernel_1850018/799403570.py:153: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 1000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.

Done.
Final output: ../data/downscaled_data/chile/sub_areas/u10m_component_downscaled_chile_m1_continental.nc
Chunk outputs:
  ../data/downscaled_data/chile/sub_areas/sub_areas/u10m_component_downscaled_chile_m1_continental_north.nc
  ../data/downscaled_data/chile/sub_areas/sub_areas/u10m_component_downscaled_chile_m1_continental_south.nc


## 9. Optional quick plot

Use this section only to visually inspect the output. If `SHAPEFILE_PATH` is `None`, the plot will show only the raster output.


In [9]:
def plot_downscaled_output(ds, variable=None, shapefile_path=None):
    """Plot the downscaled output with an optional shapefile overlay."""
    if plt is None:
        raise ImportError("matplotlib is not available in this environment.")

    if variable is None:
        variable = f"{comp}10_downscaled"

    da = ds[variable]

    try:
        da = da.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=False)
        da = da.rio.write_crs("EPSG:4326", inplace=False)
    except Exception:
        pass

    fig, ax = plt.subplots(figsize=(10, 7))
    da.plot(ax=ax)

    if shapefile_path is not None:
        if gpd is None:
            raise ImportError("geopandas is not available in this environment.")
        geometries = gpd.read_file(shapefile_path).to_crs("EPSG:4326")
        geometries.plot(ax=ax, edgecolor="black", facecolor="none")

    ax.set_title(variable)
    plt.show()


# Uncomment the next line to plot the result.
#plot_downscaled_output(output_ds, shapefile_path=SHAPEFILE_PATH)


## 10. Notes for operators

For large continental domains, increase `CONTINENTAL_N_SPLITS` if the notebook runs out of memory. For example, use `3` or `4` instead of `2`.

The output of each continental chunk is stored in `SUBAREA_OUTPUT_DIR`. The final merged output is stored in `OUTPUT_DIR`.

For island domains, the notebook creates only one output file because the bbox is processed in one piece.
